# Train `defense_impact` — no local Python needed

Runs entirely in the browser (Google Colab). Trains the SciBERT defense-impact estimator on ~10k rows and downloads the model, ready to serve.

**Runtime → Change runtime type → T4 GPU** makes embedding fast (CPU works too, just slower).

You need one file: `defense.csv` with columns `abstract`, `defense_cited` (1/0). Build it from your Scientifiq / Reliance-on-Science exports with `ml/scripts/build_defense_dataset.py`, or upload it below.

In [ ]:
!pip -q install torch transformers scikit-learn pandas numpy joblib

## 1. Upload `defense.csv`

In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()                       # pick defense.csv
path = next(iter(up))
df = pd.read_csv(path).dropna(subset=['abstract','defense_cited'])
df = df[df['abstract'].astype(str).str.len() >= 40]
# ~10k stratified sample
n, per = 10000, 5000
pos = df[df['defense_cited'] > 0.5]; neg = df[df['defense_cited'] <= 0.5]
take = lambda d,k: d.sample(min(len(d),k), random_state=42)
df = pd.concat([take(pos,per), take(neg, n-min(len(pos),per))]).sample(frac=1, random_state=42)
texts = df['abstract'].astype(str).tolist()
labels = (df['defense_cited'].astype(float) > 0.5).astype(int).tolist()
print(len(texts), 'rows,', sum(labels), 'positive')

## 2. Embed with SciBERT (frozen, mean-pooled)

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel
MODEL_ID = 'allenai/scibert_scivocab_uncased'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
enc = AutoModel.from_pretrained(MODEL_ID).to(dev).eval()
DIM = enc.config.hidden_size

def embed(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        b = [t or '' for t in texts[i:i+bs]]
        e = tok(b, padding=True, truncation=True, max_length=256, return_tensors='pt').to(dev)
        with torch.no_grad():
            h = enc(**e).last_hidden_state
        m = e['attention_mask'].unsqueeze(-1).float()
        out.append(((h*m).sum(1)/m.sum(1).clamp(min=1e-9)).cpu().numpy().astype('float32'))
        if i % (bs*20) == 0: print(f'{i}/{len(texts)}')
    return np.vstack(out)

X = embed(texts); print('embeddings', X.shape)

## 3. Fit the head (LogisticRegression) + report AUROC

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
y = np.array(labels)
Xtr,Xva,ytr,yva = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0).fit(Xtr, ytr)
p = clf.predict_proba(Xva)[:,1]
print('val AUROC', round(roc_auc_score(yva,p),3), '| AUPRC', round(average_precision_score(yva,p),3))

## 4. Save the model dir and download it

Produces `models/defense_impact/{head.joblib, meta.json}` — the exact layout `sciscore serve` expects. Download the zip, drop `models/` into `ml/`, and deploy (step 5 in `ml/README.md`).

In [ ]:
import os, json, joblib, shutil
d = 'models/defense_impact'; os.makedirs(d, exist_ok=True)
joblib.dump(clf, f'{d}/head.joblib')
json.dump({'task':'defense_impact','kind':'binary','encoder':'scibert','max_length':256,
           'dim':int(DIM),'n_train':int(len(Xtr)),'n_val':int(len(Xva)),
           'val_auroc':float(roc_auc_score(yva,p))}, open(f'{d}/meta.json','w'), indent=2)
shutil.make_archive('defense_impact_model','zip','models')
from google.colab import files as _f; _f.download('defense_impact_model.zip')
print('done')